In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, log_loss
from sklearn.utils.class_weight import compute_sample_weight

# 1. Load data
X = pd.read_csv('X_train_cleaned.csv')
y = pd.read_csv('y_train_cleaned.csv')['label']

# 2. Train/test split (80% train, 20% test) stratified
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 3. Instantiate baseline XGBoost
num_classes = len(np.unique(y))
xgb_baseline = XGBClassifier(
    objective='multi:softprob',
    num_class=num_classes,
    eval_metric='mlogloss',
    random_state=42
)

# 4. Train
xgb_baseline.fit(X_train, y_train)

# 5. Predict and evaluate
y_pred = xgb_baseline.predict(X_test)
y_proba = xgb_baseline.predict_proba(X_test)

# 6. Compute metrics
acc = accuracy_score(y_test, y_pred)
weights = compute_sample_weight(class_weight="balanced", y=y_test)
wll = log_loss(y_test, y_proba, sample_weight=weights, labels=np.unique(y_test))

print("=== XGB Baseline Performance ===")
print(f"Accuracy: {acc:.4f}")
print(f"Weighted Log Loss: {wll:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))


=== XGB Baseline Performance ===
Accuracy: 0.7693
Weighted Log Loss: 3.1249

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       0.00      0.00      0.00         1
           2       0.00      0.00      0.00         2
           3       0.60      0.23      0.33        13
           4       0.60      0.63      0.62        46
           5       0.89      0.99      0.93       875
           6       0.87      0.90      0.88       106
           7       0.50      0.19      0.28        21
           8       0.74      0.71      0.73        98
           9       0.00      0.00      0.00         5
          10       0.68      0.87      0.76       208
          11       0.73      0.67      0.70        12
          12       0.49      0.52      0.51        88
          13       0.33      0.08      0.13        12
          14       0.00      0.00      0.00        52
          15       0.75      0.60  

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, log_loss

# Load your data
X = pd.read_csv('X_train_cleaned.csv')
y = pd.read_csv('y_train_cleaned.csv')['label']

# Split into train (80%) and validation (20%)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Instantiate XGB with fixed learning_rate and high n_estimators
num_classes = len(y.unique())
xgb = XGBClassifier(
    objective='multi:softprob',
    num_class=num_classes,
    learning_rate=0.1,
    n_estimators=1000,
    eval_metric='mlogloss',
    random_state=42
)

# Fit with early stopping on the validation set
xgb.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    early_stopping_rounds=50,
    verbose=True
)

# Retrieve the best number of trees
best_n = xgb.best_iteration
print(f"Optimal n_estimators@lr=0.1: {best_n}")

# Evaluate performance
y_pred = xgb.predict(X_val)
y_proba = xgb.predict_proba(X_val)
acc = accuracy_score(y_val, y_pred)
wll = log_loss(y_val, y_proba, labels=y.unique())

print(f"\nValidation Accuracy: {acc:.4f}")
print(f"Validation Log Loss: {wll:.4f}")
print("\nClassification Report:")
print(classification_report(y_val, y_pred, zero_division=0))


TypeError: XGBClassifier.fit() got an unexpected keyword argument 'early_stopping_rounds'